---
## TB022: INTRODUCCIÓN AL DESARROLLO DE SOFTWARE
- CÓDIGO MATERIA: TB022
- AÑO, CUATRIMESTRE: 1C2026  
- CÁTEDRA: CAMEJO  
- TRABAJO: EJERCICIO ADICIONAL NO OBLIGATORIO
- Hecho con ayuda de la IA por la dificultad del mismo.
---

Lo escrito dentro de esta carpeta es sólo para explorar el Ejercicio No Obligatorio, con el fin de revisarlo en el futuro.  
Me pareció una forma de documentación adecuada en esta caso ya que no era obligatorio por su dificultad.

---

Las acciones del script son:
- `inicializar`: crea las carpetas `originales`, `procesadas` y `burlas`.
- `procesar`: valida la primera linea de cada entrega, extrae el padron y guarda el contenido sin la cabecera.
- `burlarme`: genera una version del texto reemplazando vocales por `i`.

Debajo de este resumen esta el script completo en Bash y luego la explicacion de cada parte.

Se agregan algunas decisiones de implementacion para que el comportamiento del script quede claro:
- Si faltan argumentos, el script muestra la ayuda y termina.
- Si no hay archivos `.txt`, el recorrido no intenta procesar un patron literal.
- En `procesar` se elimina solo la primera linea de cada archivo valido.
- Los mensajes estan escritos para informar al usuario en cada paso relevante.
---

SCRIPT:

In [ ]:
#!/usr/bin/env bash

set -euo pipefail

ACCION=${1:-}
DIRECTORIO=${2:-}

uso() {
  echo "Uso: bash solucion.sh <inicializar|procesar|burlarme> <directorio>"
}

listar_txt() {
  local carpeta=$1
  find "$carpeta" -maxdepth 1 -type f -name '*.txt' | sort
}

if [ -z "$ACCION" ] || [ -z "$DIRECTORIO" ]; then
  uso
  exit 1
fi

inicializar() {
  local directorio_padre=$1

  for nombre in originales procesadas burlas; do
    local directorio="$directorio_padre/$nombre"

    if [ -d "$directorio" ]; then
      echo "El directorio $directorio ya existe"
    else
      mkdir -p "$directorio"
      echo "Se creo correctamente $directorio"
    fi
  done
}

procesar() {
  local directorio=$1
  local origen="$directorio/originales"
  local destino="$directorio/procesadas"
  local regex='^Alumno: [A-Za-z]+( [A-Za-z]+)*, [A-Za-z]+( [A-Za-z]+)* - Padron: [0-9]{6}$'
  local archivos=()

  if [ ! -d "$origen" ] || [ ! -d "$destino" ]; then
    echo "Faltan directorios necesarios. Ejecuta primero la accion inicializar."
    return 1
  fi

  mapfile -t archivos < <(listar_txt "$origen")

  if [ "${#archivos[@]}" -eq 0 ]; then
    echo "No hay archivos .txt para procesar en $origen"
    return 0
  fi

  for archivo in "${archivos[@]}"; do
    local primera_linea
    local nombre_archivo
    local padron

    primera_linea=$(head -n 1 "$archivo")
    nombre_archivo=$(basename "$archivo")

    if ! printf '%s\n' "$primera_linea" | grep -qE "$regex"; then
      echo "El archivo $nombre_archivo no cumple el enunciado."
      continue
    fi

    padron=$(printf '%s\n' "$primera_linea" | grep -oE '[0-9]{6}')
    tail -n +2 "$archivo" > "$destino/${padron}.txt"
    echo "Procesamos correctamente $nombre_archivo -> ${padron}.txt"
  done
}

burlarme() {
  local directorio=$1
  local origen="$directorio/procesadas"
  local destino="$directorio/burlas"
  local archivos=()

  if [ ! -d "$origen" ] || [ ! -d "$destino" ]; then
    echo "Faltan directorios necesarios. Ejecuta primero la accion inicializar y luego procesar."
    return 1
  fi

  mapfile -t archivos < <(listar_txt "$origen")

  if [ "${#archivos[@]}" -eq 0 ]; then
    echo "No hay archivos .txt para burlarse en $origen"
    return 0
  fi

  for archivo in "${archivos[@]}"; do
    local nombre

    nombre=$(basename "$archivo")
    sed 's/[aeiou]/i/g; s/[AEIOU]/I/g' "$archivo" > "$destino/$nombre"
    echo "Burla generada: $destino/$nombre"
  done
}

case "$ACCION" in
  inicializar) inicializar "$DIRECTORIO" ;;
  procesar) procesar "$DIRECTORIO" ;;
  burlarme) burlarme "$DIRECTORIO" ;;
  *)
    echo "Error: la accion $ACCION no es conocida"
    echo "Las validas son: inicializar, procesar, burlarme"
    uso
    exit 1
    ;;
esac

exit 0

### 1) Argumentos de entrada
```bash
ACCION=${1:-}
DIRECTORIO=${2:-}

if [ -z "$ACCION" ] || [ -z "$DIRECTORIO" ]; then
```
La lectura de parametros se hace con `ACCION=${1:-}` y `DIRECTORIO=${2:-}`.

En Bash, `1` y `2` representan el primer y segundo argumento posicional. La expresion `${1:-}` usa `$1` si existe y no esta vacio; si no, lo reemplaza por una cadena vacia.

La condicion `if [ -z "$ACCION" ] || [ -z "$DIRECTORIO" ]; then` verifica si falta alguno de los dos argumentos.
- `-z` comprueba si una cadena esta vacia.
- `||` indica que alcanza con que falte uno de los dos argumentos.

Si entra en ese `if`, el script llama a `uso` y termina con `exit 1`.

### 2) Funciones auxiliares
```bash
echo "Uso: bash solucion.sh <inicializar|procesar|burlarme> <directorio>"
find "$carpeta" -maxdepth 1 -type f -name '*.txt' | sort
```
En el script aparecen dos funciones reutilizables.

`uso()` imprime la forma correcta de ejecutar el programa. La linea `echo "Uso: bash solucion.sh <inicializar|procesar|burlarme> <directorio>"` funciona como ayuda para el usuario y se llama cuando faltan argumentos o cuando la accion no es valida.

`listar_txt()` recibe una carpeta y busca los `.txt` de ese directorio. `-maxdepth 1` evita entrar en subcarpetas, `-type f` limita la busqueda a archivos, `-name '*.txt'` filtra solo archivos `.txt` y `sort` ordena la salida.

Estas dos funciones evitan repetir logica en `procesar()` y `burlarme()`.

### 3) Inicialización de carpetas
```bash
local directorio_padre=$1
for nombre in originales procesadas burlas; do
  local directorio="$directorio_padre/$nombre"
  if [ -d "$directorio" ]; then
```
`inicializar()` prepara la estructura base que usan las demas acciones.

Primero toma el directorio padre con `local directorio_padre=$1`. Despues recorre los nombres esperados con `for nombre in originales procesadas burlas; do` y arma cada ruta con `local directorio="$directorio_padre/$nombre"`.

La verificacion de existencia esta en `if [ -d "$directorio" ]; then`.
- `-d` comprueba si la ruta ya existe y es un directorio.
- Si ya existe, se informa y no se hace nada.
- Si no existe, la carpeta se crea con `mkdir -p "$directorio"`.

Con esto quedan listas las carpetas para las siguientes acciones.

### 4) Procesamiento de entregas
```bash
origen="$directorio/originales"
destino="$directorio/procesadas"

if [ ! -d "$origen" ] || [ ! -d "$destino" ]; then
mapfile -t archivos < <(listar_txt "$origen")
if [ "${#archivos[@]}" -eq 0 ]; then
```
`procesar()` transforma cada entrega original en un archivo limpio identificado por padron.

Primero define las rutas `origen="$directorio/originales"` y `destino="$directorio/procesadas"`. Despues verifica que ambas carpetas existan con `if [ ! -d "$origen" ] || [ ! -d "$destino" ]; then`.
- `! -d` significa que la carpeta no existe.
- `||` indica que si falta una de las dos, el proceso se corta con `return 1`.

Luego carga los archivos con `mapfile -t archivos < <(listar_txt "$origen")` y controla si el arreglo esta vacio con `if [ "${#archivos[@]}" -eq 0 ]; then`.
- `${#archivos[@]}` devuelve la cantidad de elementos.
- `-eq` compara numeros.

Despues, en el recorrido por archivo, la secuencia es esta:
1. `head -n 1` lee la primera linea.
2. `basename` obtiene el nombre visible.
3. `grep -qE "$regex"` valida la cabecera.
4. Si no coincide, `continue` pasa al siguiente archivo.
5. Si coincide, `grep -oE '[0-9]{6}'` extrae el padron.
6. `tail -n +2 "$archivo" > "$destino/${padron}.txt"` guarda el contenido sin la cabecera.

El resultado final es un archivo por entrega valida dentro de `procesadas`, con el cuerpo del texto sin la primera linea.

### 5) Validación del encabezado
```bash
local regex='^Alumno: [A-Za-z]+( [A-Za-z]+)*, [A-Za-z]+( [A-Za-z]+)* - Padron: [0-9]{6}$'
if ! printf '%s\n' "$primera_linea" | grep -qE "$regex"; then
```
La regla se define en `local regex='^Alumno: [A-Za-z]+( [A-Za-z]+)*, [A-Za-z]+( [A-Za-z]+)* - Padron: [0-9]{6}$'`.

La comprobacion se hace en `if ! printf '%s\n' "$primera_linea" | grep -qE "$regex"; then`.
- `grep -qE` valida con regex en modo silencioso (`-q`) y regex extendida (`-E`).
- El `!` al principio significa que entra al if cuando NO coincide.

Si no coincide, muestra mensaje y usa `continue`, que salta al siguiente archivo del `for` sin cortar todo el proceso.

### 6) Generación de burlas
```bash
origen="$directorio/procesadas"
destino="$directorio/burlas"

if [ ! -d "$origen" ] || [ ! -d "$destino" ]; then
mapfile -t archivos < <(listar_txt "$origen")
sed 's/[aeiou]/i/g; s/[AEIOU]/I/g' "$archivo" > "$destino/$nombre"
```
`burlarme()` toma los archivos ya procesados y genera una version transformada en `burlas`.

Primero define las rutas `origen="$directorio/procesadas"` y `destino="$directorio/burlas"`. Luego verifica que ambas carpetas existan con `if [ ! -d "$origen" ] || [ ! -d "$destino" ]; then`.
- `! -d` significa que no existe el directorio.
- `||` indica que si falta una de las dos carpetas, el proceso se corta con `return 1`.

Despues carga los archivos con `mapfile -t archivos < <(listar_txt "$origen")` y, si no hay ninguno, termina con `return 0`.

En el recorrido por archivo, la secuencia es esta:
1. `basename` obtiene el nombre del archivo.
2. `sed 's/[aeiou]/i/g; s/[AEIOU]/I/g'` reemplaza las vocales.
3. La salida se redirige a `"$destino/$nombre"`.

Ese `sed` hace dos reemplazos:
- `s/[aeiou]/i/g` cambia vocales minusculas por `i`.
- `s/[AEIOU]/I/g` cambia vocales mayusculas por `I`.

El resultado conserva el mismo nombre de archivo y se guarda en `burlas`.

### 7) Selección de acción
```bash
case "$ACCION" in
inicializar) inicializar "$DIRECTORIO" ;;
procesar) procesar "$DIRECTORIO" ;;
burlarme) burlarme "$DIRECTORIO" ;;
```
`case "$ACCION" in` elige que funcion ejecutar.

Las ramas principales son `inicializar`, `procesar` y `burlarme`, y cada una llama directamente a su funcion correspondiente.
El caso `*)` es el valor por defecto y cubre cualquier accion no contemplada. Si ocurre, el script muestra el error, vuelve a imprimir la ayuda con `uso` y termina con `exit 1`.